# Quant Learning Guide

这个 notebook 是你的学习主线。每个 Phase 包含：
1. **核心概念** — 必须理解的知识点
2. **自测问题** — 检验是否理解
3. **代码实验** — 动手跑一跑
4. **延伸阅读** — 可选深入学习

---
## Phase 1 — Data Layer 核心概念

### 1. OHLCV 是什么？

每根 K 线由 5 个数据组成：

| 字段 | 含义 | 说明 |
|------|------|------|
| **O**pen | 开盘价 | 这根K线开始时的价格 |
| **H**igh | 最高价 | 这根K线期间的最高价格 |
| **L**ow | 最低价 | 这根K线期间的最低价格 |
| **C**lose | 收盘价 | 这根K线结束时的价格 |
| **V**olume | 成交量 | 这根K线期间的交易量 |

**为什么是这五个？**
- Open/Close 代表市场情绪的起点和终点
- High/Low 代表波动范围
- Volume 代表市场参与度

In [ ]:
# 实验：下载并查看 OHLCV 数据
import yfinance as yf

# 下载苹果股票最近5天的数据
df = yf.download("AAPL", period="5d")
df

### 2. Adjusted Close 是什么？

**Close** = 当天实际收盘价格

**Adjusted Close (Adj Close)** = 调整后的收盘价

为什么要调整？
- 股票会发生**分红**（dividend）和**拆股**（split）
- 这些事件会让价格"跳变"，但不是真正的涨跌
- Adj Close 把历史价格调整到同一基准，方便计算真实收益

**例子：**
- 某股票 $100，1拆10后变成 $10
- Close 会显示 $10（看起来跌了90%）
- Adj Close 会把之前的价格都除以10，保持连续性

In [ ]:
# 实验：对比 Close 和 Adj Close
df = yf.download("AAPL", period="5y")

# 查看 Close 和 Adj Close 的差异
print("最近5天 Close vs Adj Close:")
print(df[['Close', 'Adj Close']].tail())

# 计算两种方式的收益率差异
close_return = (df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100
adj_return = (df['Adj Close'].iloc[-1] / df['Adj Close'].iloc[0] - 1) * 100

print(f"\n按 Close 计算的5年收益率: {close_return:.1f}%")
print(f"按 Adj Close 计算的5年收益率: {adj_return:.1f}%")

### 3. ETF vs 股票

| | 股票 (Stock) | ETF |
|---|-------------|-----|
| 定义 | 单家公司的股份 | 一篮子股票/资产的基金 |
| 风险 | 取决于单家公司 | 分散在一篮子资产 |
| 例子 | AAPL, TSLA, NVDA | SPY, QQQ, VTI |

**本项目默认关注的 ETF：**

| Ticker | 名称 | 跟踪 |
|--------|------|------|
| SPY | 标普500 ETF | 美国500家大公司 |
| QQQ | 纳斯达克100 ETF | 科技股为主 |
| DIA | 道琼斯 ETF | 30家工业龙头 |
| IWM | 罗素2000 ETF | 小盘股 |
| VIX | 波动率指数 | 市场恐慌程度 |

**为什么先看 ETF？**
- 代表整体市场，不是单个公司
- 适合建立市场直觉
- 风险更分散

### 4. 为什么需要缓存？

**问题：**
- 每次运行都要下载数据 → 慢
- API 有请求限制 → 可能被封
- 网络不稳定 → 程序崩溃

**解决方案：缓存**
1. 第一次下载后存到本地（parquet 文件）
2. 后续读取本地文件
3. 设置过期时间（如24小时），过期后重新下载

**数据流：**
```
请求数据 → 检查缓存是否存在且未过期
                ↓ 是                    ↓ 否
           读取本地缓存            从 API 下载
                                       ↓
                                   保存到缓存
```

### 自测问题（Phase 1）

完成以下问题后，再让 Codex 写代码：

1. 如果你要看苹果股票今天的波动范围，你会看哪两个字段？

2. 为什么要用 Adj Close 而不是 Close 计算历史收益？

3. SPY 和 AAPL 的区别是什么？为什么新手更适合先研究 SPY？

4. 如果你的程序每天自动运行，缓存应该设多久过期？为什么？

---

**完成后在下面记录你的答案：**

In [ ]:
# 你的答案
answer_1 = "High 和 Low"  # 替换成你的答案
answer_2 = ""
answer_3 = ""
answer_4 = ""

---
## Phase 2 — Visualization 核心概念

### 1. K线图（Candlestick Chart）

每根K线包含四个价格：Open, High, Low, Close

**红色/绿色（或红/绿）：**
- 绿色（阳线）：Close > Open，价格上涨
- 红色（阴线）：Close < Open，价格下跌

**阅读K线：**
- 上影线 = High - max(Open, Close)
- 下影线 = min(Open, Close) - Low
- 实体 = |Close - Open|

影线越长，说明该方向遇到阻力越大。

In [ ]:
# 实验：画一根K线理解结构
import plotly.graph_objects as go

# 一根示例K线
fig = go.Figure(data=[go.Candlestick(
    x=['Day 1'],
    open=[100],
    high=[110],
    low=[95],
    close=[105]
)])
fig.update_layout(title='单根K线示例：Open=100, High=110, Low=95, Close=105', height=400)
fig.show()

### 2. 成交量（Volume）的意义

**价涨量增** = 上涨有支撑，趋势可能延续
**价涨量缩** = 上涨动力不足，可能回调
**价跌量增** = 抛售压力大，可能继续跌
**价跌量缩** = 抛售减弱，可能企稳

成交量验证价格趋势的可信度。

### 3. 多股对比与归一化

问题：SPY 价格 $400，QQQ 价格 $350，直接画线对比没意义

解决：**归一化** = 把起点都设为 100 或 1

```
归一化价格 = 当前价格 / 起始价格 × 100
```

这样可以看到谁的涨幅更大。

In [ ]:
# 实验：归一化对比
import pandas as pd

# 模拟数据
spy_prices = [400, 410, 420, 415, 430]
qqq_prices = [350, 365, 380, 370, 390]

# 归一化
spy_normalized = [p / spy_prices[0] * 100 for p in spy_prices]
qqq_normalized = [p / qqq_prices[0] * 100 for p in qqq_prices]

print("SPY 归一化:", spy_normalized)
print("QQQ 归一化:", qqq_normalized)
print("\nSPY 涨幅:", f"{spy_normalized[-1] - 100:.1f}%")
print("QQQ 涨幅:", f"{qqq_normalized[-1] - 100:.1f}%")

### 自测问题（Phase 2）

1. 一根绿色K线，上影线很长，下影线很短，说明了什么？

2. 如果股价上涨但成交量持续萎缩，你怎么看？

3. 为什么要归一化后再比较多只股票？

---
## Phase 3 — Technical Indicators 核心概念

### 1. 移动平均线（Moving Average, MA）

**定义：** 过去 N 天价格的平均值，每天滚动计算

常用周期：
- MA20 = 过去20天均价（约1个月）
- MA50 = 过去50天均价（约2.5个月）
- MA200 = 过去200天均价（约1年）

**用法：**
- 价格在 MA 之上 = 趋势偏多
- 价格在 MA 之下 = 趋势偏空
- MA 交叉 = 可能趋势反转（短期 MA 上穿长期 MA = 金叉，看多）

**重要：** MA 是滞后指标，反映过去，不是预测未来

In [ ]:
# 实验：手动计算 MA
prices = [100, 102, 101, 103, 105, 104, 106, 108, 107, 109]

# 计算 5日均线
ma5 = []
for i in range(len(prices)):
    if i < 4:  # 前4天数据不够
        ma5.append(None)
    else:
        ma5.append(sum(prices[i-4:i+1]) / 5)

print("价格:", prices)
print("MA5:", ma5)

### 2. RSI（相对强弱指标）

**定义：** 衡量一段时间内涨跌幅的比例，范围 0-100

**解读：**
- RSI > 70 = 超买（可能回调）
- RSI < 30 = 超卖（可能反弹）
- RSI ≈ 50 = 多空平衡

**公式简化理解：**
```
RSI = 100 × 平均涨幅 / (平均涨幅 + 平均跌幅)
```

**注意：** 超买不等于马上跌，超卖不等于马上涨

### 3. MACD（异同移动平均线）

**组成：**
- MACD 线 = 快线(12日EMA) - 慢线(26日EMA)
- Signal 线 = MACD 的 9日EMA
- Histogram = MACD - Signal

**用法：**
- MACD 上穿 Signal = 金叉，看多
- MACD 下穿 Signal = 死叉，看空
- Histogram 为正 = 多头动能
- Histogram 为负 = 空头动能

### 4. Bollinger Bands（布林带）

**组成：**
- 中轨 = 20日均线
- 上轨 = 中轨 + 2倍标准差
- 下轨 = 中轨 - 2倍标准差

**用法：**
- 价格触及上轨 = 可能超买
- 价格触及下轨 = 可能超卖
- 带宽收窄 = 波动率降低，可能有大行情
- 带宽扩张 = 波动率增加

### 5. ATR（平均真实波幅）

**定义：** 衡量波动程度的绝对值

**计算：**
```
True Range = max(High-Low, |High-昨日Close|, |Low-昨日Close|)
ATR = True Range 的 14日平均值
```

**用途：**
- 评估股票波动性（ATR 越大越波动）
- 设定止损位（如：止损 = 入场价 - 2×ATR）
- 仓位管理（波动大的股票买少一点）

### 自测问题（Phase 3）

1. MA20 上穿 MA50 叫什么？通常意味着什么？

2. RSI=85 说明什么？能不能说马上要跌？

3. 布林带收窄（带宽变小）说明了什么？

4. 为什么说技术指标不是预测工具？它们的作用是什么？

---
## 学习检查清单

完成以下步骤后，再进入下一 Phase：

- [ ] 阅读本 Phase 的概念说明
- [ ] 运行所有代码单元格
- [ ] 回答自测问题
- [ ] 让 Codex 写代码
- [ ] 读 Codex 写的代码，不懂的问我
- [ ] 自己改参数做实验

**准备好后，切换到 Codex 执行对应 Phase 的任务。**